# Model 1) google/muril-base-cased on hindi dataset

In [1]:
!pip install torch torchvision torchaudio pytorch-lightning scikit-learn pandas numpy sentencepiece accelerate
!pip install transformers==4.41.2

In [2]:
#!/usr/bin/env python3
"""
MuRIL Hindi Stance Classification
Dataset files:
    train_hindi.csv
    valid_hindi.csv
    test_hindi.csv
Columns:
    text,label
Labels:
    favor, against, neutral
"""

import os
import random
import numpy as np
import pandas as pd
import torch
import pytorch_lightning as pl

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

from torch.utils.data import Dataset, DataLoader
from torch import nn

from transformers import (
    AutoTokenizer,
    AutoModel,
    AdamW,
    get_linear_schedule_with_warmup,
)

from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from torchmetrics.classification import MulticlassF1Score


In [4]:
SEED = 42
MAX_LEN = 128
BATCH_SIZE = 16

# Train for more epochs; EarlyStopping will stop automatically
EPOCHS = 20

LR = 2e-5
MODEL_NAME = "google/muril-base-cased"

pl.seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_df = pd.read_csv("train_hindi.csv")
valid_df = pd.read_csv("valid_hindi.csv")
test_df = pd.read_csv("test_hindi.csv")

train_df = train_df.dropna()
valid_df = valid_df.dropna()
test_df = test_df.dropna()

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["label"])
valid_df["label"] = label_encoder.transform(valid_df["label"])
test_df["label"] = label_encoder.transform(test_df["label"])

NUM_CLASSES = len(label_encoder.classes_)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HindiDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_loader = DataLoader(HindiDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(HindiDataset(valid_df), batch_size=BATCH_SIZE)
test_loader = DataLoader(HindiDataset(test_df), batch_size=BATCH_SIZE)

class MurilClassifier(pl.LightningModule):

    def __init__(self):
        super().__init__()

        self.bert = AutoModel.from_pretrained(MODEL_NAME)

        self.drop = nn.Dropout(0.3)

        self.fc = nn.Linear(self.bert.config.hidden_size, NUM_CLASSES)

        self.loss_fn = nn.CrossEntropyLoss()

        self.val_f1 = MulticlassF1Score(
            num_classes=NUM_CLASSES,
            average="macro"
        )

        self.test_f1 = MulticlassF1Score(
            num_classes=NUM_CLASSES,
            average="macro"
        )

    def forward(self, ids, mask):
        output = self.bert(
            input_ids=ids,
            attention_mask=mask
        )

        pooled = output.last_hidden_state[:, 0]

        pooled = self.drop(pooled)

        logits = self.fc(pooled)

        return logits

    def training_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.loss_fn(
            logits,
            batch["labels"]
        )

        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_epoch=True
        )

        return loss

    def validation_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.loss_fn(
            logits,
            batch["labels"]
        )

        preds = torch.argmax(logits, dim=1)

        f1 = self.val_f1(
            preds,
            batch["labels"]
        )

        self.log(
            "val_loss",
            loss,
            prog_bar=False,
            on_epoch=True
        )

        self.log(
            "val_f1",
            f1,
            prog_bar=True,
            on_epoch=True,
            on_step=False
        )

    def test_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.loss_fn(
            logits,
            batch["labels"]
        )

        preds = torch.argmax(logits, dim=1)

        f1 = self.test_f1(
            preds,
            batch["labels"]
        )

        self.log(
            "test_loss",
            loss
        )

        self.log(
            "test_f1",
            f1,
            prog_bar=True,
            on_epoch=True,
            on_step=False
        )

    def configure_optimizers(self):

        optimizer = AdamW(
            self.parameters(),
            lr=LR
        )

        total_steps = len(train_loader) * EPOCHS

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }

model=MurilClassifier()

checkpoint = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    filename="best_muril_hindi",
    save_top_k=1
)

early = EarlyStopping(
    monitor="val_f1",
    mode="max",
    patience=4,
    verbose=True
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else 32,
    callbacks=[checkpoint, early],
    log_every_n_steps=10
)

INFO:lightning_fabric.utilities.seed:Seed set to 42
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable L

In [5]:
trainer.fit(model,train_loader,valid_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert    │ BertModel         │  237 M │ eval  │     0 │
│ 1 │ drop    │ Dropout           │      0 │ train │     0 │
│ 2 │ fc      │ Linear            │  2.3 K │ train │     0 │
│ 3 │ loss_fn │ CrossEntropyLoss  │      0 │ train │     0 │
│ 4 │ val_f1  │ MulticlassF1Score │      0 │ train │     0 │
│ 5 │ test_f1 │ MulticlassF1Score │      0 │ train │     0 │
└───┴─────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 237 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 237 M                                                                                                
Total estimated model params size (MB): 950.234                                                                    
Modules in train mode: 5                                                                                           
Modules in eval mode: 228                                                                                          
Total FLOPs: 0

Output()

In [6]:
best = MurilClassifier.load_from_checkpoint(
    checkpoint.best_model_path
)

trainer.test(best,test_loader)

best.eval()
best.to(device)

preds=[]
truth=[]

with torch.no_grad():
    for batch in test_loader:
        ids=batch["input_ids"].to(device)
        mask=batch["attention_mask"].to(device)
        labels=batch["labels"].numpy()

        logits=best(ids,mask)
        p=torch.argmax(logits,1).cpu().numpy()

        preds.extend(p)
        truth.extend(labels)

acc=accuracy_score(truth,preds)
prec,rec,f1,_=precision_recall_fscore_support(truth,preds,average="macro")

print("\nAccuracy :",acc)
print("Precision:",prec)
print("Recall   :",rec)
print("Macro F1 :",f1)

print("\nClassification Report\n")
print(classification_report(
    truth,
    preds,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix\n")
print(confusion_matrix(truth,preds))

torch.save(best.state_dict(),"muril_hindi_model.pt")

print("\nModel saved as muril_hindi_model.pt")

Testing ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31/31 0:00:01 • 0:00:00 25.41it/s

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_f1          │    0.5778653025627136     │
│         test_loss         │    0.9550490379333496     │
└───────────────────────────┴───────────────────────────┘


Accuracy : 0.5959595959595959
Precision: 0.6024442817899395
Recall   : 0.5965537431288604
Macro F1 : 0.5945183727884681

Classification Report

              precision    recall  f1-score   support

     against       0.66      0.55      0.60       177
       favor       0.59      0.55      0.57       155
     neutral       0.55      0.69      0.61       163

    accuracy                           0.60       495
   macro avg       0.60      0.60      0.59       495
weighted avg       0.60      0.60      0.60       495


Confusion Matrix

[[ 97  28  52]
 [ 29  85  41]
 [ 20  30 113]]

Model saved as muril_hindi_model.pt


# Model 2) google/muril-base-cased on bengali dataset

In [3]:
SEED = 42
MAX_LEN = 128
BATCH_SIZE = 16

# Train for more epochs; EarlyStopping will stop automatically
EPOCHS = 20

LR = 2e-5
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"

pl.seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_df = pd.read_csv("train_bengali.csv")
valid_df = pd.read_csv("valid_bengali.csv")
test_df = pd.read_csv("test_bengali.csv")

train_df = train_df.dropna()
valid_df = valid_df.dropna()
test_df = test_df.dropna()

label_encoder = LabelEncoder()

train_df["label"] = label_encoder.fit_transform(train_df["label"])
valid_df["label"] = label_encoder.transform(valid_df["label"])
test_df["label"] = label_encoder.transform(test_df["label"])

NUM_CLASSES = len(label_encoder.classes_)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class HindiDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_loader = DataLoader(HindiDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(HindiDataset(valid_df), batch_size=BATCH_SIZE)
test_loader = DataLoader(HindiDataset(test_df), batch_size=BATCH_SIZE)

class MurilClassifier(pl.LightningModule):

    def __init__(self):
        super().__init__()

        self.bert = AutoModel.from_pretrained(MODEL_NAME)

        self.drop = nn.Dropout(0.3)

        self.fc = nn.Linear(self.bert.config.hidden_size, NUM_CLASSES)

        self.loss_fn = nn.CrossEntropyLoss()

        self.val_f1 = MulticlassF1Score(
            num_classes=NUM_CLASSES,
            average="macro"
        )

        self.test_f1 = MulticlassF1Score(
            num_classes=NUM_CLASSES,
            average="macro"
        )

    def forward(self, ids, mask):
        output = self.bert(
            input_ids=ids,
            attention_mask=mask
        )

        pooled = output.last_hidden_state[:, 0]

        pooled = self.drop(pooled)

        logits = self.fc(pooled)

        return logits

    def training_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.loss_fn(
            logits,
            batch["labels"]
        )

        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_epoch=True
        )

        return loss

    def validation_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.loss_fn(
            logits,
            batch["labels"]
        )

        preds = torch.argmax(logits, dim=1)

        f1 = self.val_f1(
            preds,
            batch["labels"]
        )

        self.log(
            "val_loss",
            loss,
            prog_bar=False,
            on_epoch=True
        )

        self.log(
            "val_f1",
            f1,
            prog_bar=True,
            on_epoch=True,
            on_step=False
        )

    def test_step(self, batch, batch_idx):

        logits = self(
            batch["input_ids"],
            batch["attention_mask"]
        )

        loss = self.loss_fn(
            logits,
            batch["labels"]
        )

        preds = torch.argmax(logits, dim=1)

        f1 = self.test_f1(
            preds,
            batch["labels"]
        )

        self.log(
            "test_loss",
            loss
        )

        self.log(
            "test_f1",
            f1,
            prog_bar=True,
            on_epoch=True,
            on_step=False
        )

    def configure_optimizers(self):

        optimizer = AdamW(
            self.parameters(),
            lr=LR
        )

        total_steps = len(train_loader) * EPOCHS

        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }

model=MurilClassifier()

checkpoint = ModelCheckpoint(
    monitor="val_f1",
    mode="max",
    filename="best_muril_hindi",
    save_top_k=1
)

early = EarlyStopping(
    monitor="val_f1",
    mode="max",
    patience=4,
    verbose=True
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else 32,
    callbacks=[checkpoint, early],
    log_every_n_steps=10
)

INFO:lightning_fabric.utilities.seed:Seed set to 42
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable L

In [4]:
trainer.fit(model,train_loader,valid_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert    │ BertModel         │  278 M │ eval  │     0 │
│ 1 │ drop    │ Dropout           │      0 │ train │     0 │
│ 2 │ fc      │ Linear            │  4.6 K │ train │     0 │
│ 3 │ loss_fn │ CrossEntropyLoss  │      0 │ train │     0 │
│ 4 │ val_f1  │ MulticlassF1Score │      0 │ train │     0 │
│ 5 │ test_f1 │ MulticlassF1Score │      0 │ train │     0 │
└───┴─────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 278 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 278 M                                                                                                
Total estimated model params size (MB): 1,112.184                                                                  
Modules in train mode: 5                                                                                           
Modules in eval mode: 228                                                                                          
Total FLOPs: 0

Output()

In [5]:
best = MurilClassifier.load_from_checkpoint(
    checkpoint.best_model_path
)

trainer.test(best,test_loader)

best.eval()
best.to(device)

preds=[]
truth=[]

with torch.no_grad():
    for batch in test_loader:
        ids=batch["input_ids"].to(device)
        mask=batch["attention_mask"].to(device)
        labels=batch["labels"].numpy()

        logits=best(ids,mask)
        p=torch.argmax(logits,1).cpu().numpy()

        preds.extend(p)
        truth.extend(labels)

acc=accuracy_score(truth,preds)
prec,rec,f1,_=precision_recall_fscore_support(truth,preds,average="macro")

print("\nAccuracy :",acc)
print("Precision:",prec)
print("Recall   :",rec)
print("Macro F1 :",f1)

print("\nClassification Report\n")
print(classification_report(
    truth,
    preds,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix\n")
print(confusion_matrix(truth,preds))

torch.save(best.state_dict(),"muril_hindi_model.pt")

print("\nModel saved as muril_hindi_model.pt")

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()